In [ ]:

import os
import pygrib
import numpy as np
import pandas as pd
from haversine import haversine, Unit

# Directories
base_directory = '/home/jaguir26/projects/Project/Input/GLOFAS/Medium_Range'
output_file = '/home/jaguir26/projects/Project/Input/GLOFAS/consolidated_glofas_data.csv'

# Initialize an empty list to store the consolidated data
data_list = []

def process_glofas_data(grbs, closest_indices):
    if grbs is None or closest_indices is None:
        print("GRIB file or closest indices not provided.")
        return []

    closest_i, closest_j = closest_indices
    glofas_data = []

    for grb in grbs:
        lead_time = grb['forecastTime']
        ensemble_member = grb['perturbationNumber']
        data_value = grb.values[closest_i, closest_j]
        glofas_data.append((lead_time, ensemble_member, data_value))

    grbs.close()
    return glofas_data

def convert_longitude_to_neg_180_180(lon):
    if lon > 180:
        return lon - 360
    return lon

def convert_longitude_to_0_360(lon):
    if lon < 0:
        return lon + 360
    return lon

def find_closest_coordinates(grbs, target_location):
    if grbs is None:
        return None, None, None, None

    min_distance = float('inf')
    closest_coordinates = None
    closest_i = None
    closest_j = None

    for grb in grbs:
        if grb['perturbationNumber'] == 0:
            latitudes, longitudes = grb.latlons()
            for i in range(len(latitudes)):
                for j in range(len(longitudes[0])):
                    coord = (latitudes[i][j], convert_longitude_to_neg_180_180(longitudes[i][j]))
                    distance = haversine(target_location, coord, unit=Unit.METERS)
                    if distance < min_distance:
                        min_distance = distance
                        closest_coordinates = coord
                        closest_i = i
                        closest_j = j
            break

    closest_coordinates_glofas = None
    if closest_coordinates:
        closest_coordinates_glofas = (closest_coordinates[0], convert_longitude_to_0_360(closest_coordinates[1]))
    
    return closest_coordinates_glofas, closest_i, closest_j, min_distance

# Main processing loop
# target_location = (34.05, -118.25)  # Example coordinates, replace with actual coordinates
# target_location = (37.449999999999996, -122.05000000000004)
target_location = (37.0443931, -122.072464)
for date_folder in sorted(os.listdir(base_directory)):
    date_folder_path = os.path.join(base_directory, date_folder)
    if not os.path.isdir(date_folder_path):
        continue

    for file_name in os.listdir(date_folder_path):
        if file_name.endswith('.grib'):
            file_path = os.path.join(date_folder_path, file_name)
            grbs = pygrib.open(file_path)
            closest_coordinates_glofas, closest_i, closest_j, min_distance = find_closest_coordinates(grbs, target_location)
            if closest_coordinates_glofas:
                print(f"Processing file {file_name} for date {date_folder}")
                glofas_data = process_glofas_data(grbs, (closest_i, closest_j))

                # Append data to the list
                for lead_time, ensemble_member, discharge in glofas_data:
                    data_list.append({
                        'start_forecast': date_folder,
                        'lead_time': lead_time,
                        'ensemble_member': ensemble_member,
                        'discharge': discharge
                    })

# Convert the list to a DataFrame
consolidated_df = pd.DataFrame(data_list)

# # Save the consolidated DataFrame to a CSV file
# consolidated_df.to_csv(output_file, index=False)
# print(f"Consolidated data saved to {output_file}")

In [ ]:
import os
import pygrib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from haversine import haversine, Unit
import dataretrieval.nwis as nwis
from datetime import datetime, timedelta

# Load the consolidated DataFrame
consolidated_df = pd.read_csv('/home/jaguir26/projects/Project/Input/GLOFAS/consolidated_glofas_data.csv')

# Ensure 'start_forecast' column is in datetime format
consolidated_df['start_forecast'] = pd.to_datetime(consolidated_df['start_forecast'])


In [ ]:

# # Constants
# CFSToCMS_CONVERSION_FACTOR = 0.0283168466

# # Define time range
# start_usgs = '1979-01-01'
# today = datetime.today()
# end_usgs = today.strftime('%Y-%m-%d')

# # Fetch daily data for the site
# site_code = '11160500'
# df = nwis.get_record(sites=site_code, service='dv', parameterCd='00060', statCd='00003', start=start_usgs, end=end_usgs)

# # Log-transform the flow data; we add 1 to handle cases where the value is 0
# df['log_discharge'] = np.log(df['00060_Mean'].astype(float) + 1)
# # Keep only the relevant column
# df = df[['log_discharge']]
# # Reverse the log transformation to get back the discharge in cfs
# df['discharge_cfs'] = np.exp(df['log_discharge']) - 1
# # Convert discharge from cfs to cms
# df['discharge_cms'] = df['discharge_cfs'] * CFSToCMS_CONVERSION_FACTOR
# # Optionally, log-transform the discharge in cms
# df['log_discharge_cms'] = np.log(df['discharge_cms'] + 1)

# # Fetch metadata for the USGS site
# site_info = nwis.get_record(sites=site_code, service='site')
# station_name = site_info['station_nm'][0]

# # Extract latitude and longitude
# latitude = float(site_info['dec_lat_va'][0])
# longitude = float(site_info['dec_long_va'][0])

# # Combine into a single target_location tuple
# target_location = (latitude, longitude)
# target_lat, target_lon = target_location
# print(f"The coordinates for site {site_code} are {target_location}")

# # Define the consistent date range for the x-axis
# start_date = consolidated_df['start_forecast'].min()
# end_date = consolidated_df['start_forecast'].max() + pd.to_timedelta(consolidated_df['lead_time'].max(), unit='h')

# # Plotting function
# def plot_forecasts(df, start_forecast, start_date, end_date, usgs_df):
#     plt.figure(figsize=(20, 6))
    
#     # Filter the DataFrame for the specific start_forecast date
#     subset = df[df['start_forecast'] == start_forecast]
    
#     # Generate the x-axis dates
#     start_forecast_date = pd.Timestamp(start_forecast)
#     lead_times_hours = np.arange(24, 721, 24)  # Assuming 24-hour intervals
#     lead_times_timedelta = pd.to_timedelta(lead_times_hours, unit='h')
#     ensemble_timestamps = start_forecast_date + lead_times_timedelta
    
#     # Plot each ensemble member
#     for ensemble_member in subset['ensemble_member'].unique():
#         member_data = subset[subset['ensemble_member'] == ensemble_member].set_index('lead_time').reindex(lead_times_hours).sort_index()
#         plt.plot(ensemble_timestamps, np.log(member_data['discharge'] + 1), color='orange', alpha=0.2)
    
#     # Plot mean ensemble
#     mean_ensemble = subset.groupby('lead_time')['discharge'].mean().reindex(lead_times_hours)
#     plt.plot(ensemble_timestamps, np.log(mean_ensemble + 1), color='darkorange', linewidth=2.5, label='Mean Ensemble')
    
#     # Plot USGS discharge data
#     plt.plot(usgs_df.index, usgs_df['log_discharge_cms'], marker='o', linestyle='--', color='forestgreen', markersize=2, alpha=1, label='USGS')
    
#     plt.xlabel('Time')
#     plt.ylabel('Log Discharge (cms)')
#     plt.title(f'Ensemble Members and Mean Ensemble - {start_forecast} with USGS Data - {station_name}')
#     plt.grid(True)
#     plt.xlim([start_date, end_date])
#     plt.legend()
#     plt.show()

# # Get unique start_forecast dates
# unique_start_forecasts = consolidated_df['start_forecast'].unique()

# # Plot forecasts for each start_forecast date
# for start_forecast in unique_start_forecasts:
#     plot_forecasts(consolidated_df, start_forecast, start_date, end_date, df)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dataretrieval.nwis as nwis
from datetime import datetime, timedelta

# Ensure 'start_forecast' column is in datetime format
consolidated_df['start_forecast'] = pd.to_datetime(consolidated_df['start_forecast'])

# Calculate target dates and subtract one day
consolidated_df['target_date'] = consolidated_df['start_forecast'] + pd.to_timedelta(consolidated_df['lead_time'], unit='h') - timedelta(days=1)

# Apply log(x + 1) transformation to the discharge values
consolidated_df['log_discharge'] = np.log(consolidated_df['discharge'] + 1)

# Function to calculate the weighted average of the log-transformed values
def weighted_avg(group, power):
    lead_times = group['lead_time']
    log_discharges = group['log_discharge']
    
    # Ensure lead times are not zero to avoid division by zero
    if (lead_times == 0).any():
        lead_times = lead_times.replace(0, 1)
    
    weights = lead_times ** power
    weights /= weights.sum()
    
    weighted_sum = (weights * log_discharges).sum()
    return pd.Series(weighted_sum, index=['weighted_average_log_discharge'])

# Calculate weighted averages for all ensemble members with a specified power for weights
def calculate_weighted_averages(df, power):
    grouped = df.groupby(['target_date', 'ensemble_member'])
    weighted_averages = grouped.apply(weighted_avg, power=power).reset_index()
    return weighted_averages

# Specify the power for weight calculation
power = -1.001  # Example: weights proportional to (lead_times)^0.1

# Calculate weighted averages
weighted_averages_all = calculate_weighted_averages(consolidated_df, power)

# Constants for USGS data
CFSToCMS_CONVERSION_FACTOR = 0.0283168466

# Define time range
start_usgs = '1979-01-01'
today = datetime.today()
end_usgs = today.strftime('%Y-%m-%d')

# Fetch daily data for the site
site_code = '11160500'
df_usgs = nwis.get_record(sites=site_code, service='dv', parameterCd='00060', statCd='00003', start=start_usgs, end=end_usgs)

# Log-transform the flow data; we add 1 to handle cases where the value is 0
df_usgs['log_discharge'] = np.log(df_usgs['00060_Mean'].astype(float) + 1)
# Keep only the relevant column
df_usgs = df_usgs[['log_discharge']]
# Reverse the log transformation to get back the discharge in cfs
df_usgs['discharge_cfs'] = np.exp(df_usgs['log_discharge']) - 1
# Convert discharge from cfs to cms
df_usgs['discharge_cms'] = df_usgs['discharge_cfs'] * CFSToCMS_CONVERSION_FACTOR
# Optionally, log-transform the discharge in cms
df_usgs['log_discharge_cms'] = np.log(df_usgs['discharge_cms'] + 1)

# Fetch metadata for the USGS site
site_info = nwis.get_record(sites=site_code, service='site')
station_name = site_info['station_nm'][0]

# Extract latitude and longitude
latitude = float(site_info['dec_lat_va'][0])
longitude = float(site_info['dec_long_va'][0])

# Combine into a single target_location tuple
target_location = (latitude, longitude)
target_lat, target_lon = target_location
print(f"The coordinates for site {site_code} are {target_location}")

# Define the consistent date range for the x-axis
start_date = weighted_averages_all['target_date'].min()
end_date = weighted_averages_all['target_date'].max()

# Create a DataFrame to store all the time series resulting from the weighting
weighted_time_series_df = weighted_averages_all.pivot(index='target_date', columns='ensemble_member', values='weighted_average_log_discharge')
weighted_time_series_df = weighted_time_series_df.reset_index()

# # Save the DataFrame to a CSV file
# output_csv_path = '/home/jaguir26/project1_ucsc_phd/weighted_time_series.csv'
# weighted_time_series_df.to_csv(output_csv_path, index=False)

# print(f"Weighted time series saved to {output_csv_path}")


In [ ]:

# Plotting the weighted average time series for all ensemble members along with USGS data
plt.figure(figsize=(20, 10))
ensemble_members = weighted_averages_all['ensemble_member'].unique()

for ensemble_member in ensemble_members:
    member_data = weighted_averages_all[weighted_averages_all['ensemble_member'] == ensemble_member]
    plt.plot(member_data['target_date'], member_data['weighted_average_log_discharge'], linestyle='-', alpha=0.5, label=f'Ensemble {ensemble_member}',color='orange')

# Plot USGS discharge data
plt.plot(df_usgs.index, df_usgs['log_discharge_cms'], marker='o', linestyle='--', color='forestgreen', markersize=2, alpha=1, label='USGS')

plt.xlabel('Time')
plt.ylabel('Log Discharge (cms)')
plt.title('Weighted Average Log Discharge for All Ensemble Members with USGS Data')
plt.grid(True)
plt.xlim([start_date, end_date])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()



In [ ]:
import os
import pygrib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from haversine import haversine, Unit
import dataretrieval.nwis as nwis
from datetime import datetime, timedelta

# # Load the consolidated DataFrame
# consolidated_df = pd.read_csv('/home/jaguir26/projects/Project/Input/GLOFAS/consolidated_glofas_data.csv')

# Ensure 'start_forecast' column is in datetime format
consolidated_df['start_forecast'] = pd.to_datetime(consolidated_df['start_forecast'])

# Set the cutoff date
cutoff_date = '2022-12-25'

# Filter the DataFrame based on the cutoff date
consolidated_df = consolidated_df[consolidated_df['start_forecast'] <= cutoff_date]

# Calculate target dates and subtract one day
consolidated_df['target_date'] = consolidated_df['start_forecast'] + pd.to_timedelta(consolidated_df['lead_time'], unit='h') - timedelta(days=1)

# Apply log(x + 1) transformation to the discharge values
consolidated_df['log_discharge'] = np.log(consolidated_df['discharge'] + 1)

# Function to calculate the weighted average of the log-transformed values
def weighted_avg(group, power):
    lead_times = group['lead_time']
    log_discharges = group['log_discharge']
    
    # Ensure lead times are not zero to avoid division by zero
    if (lead_times == 0).any():
        lead_times = lead_times.replace(0, 1)
    
    weights = lead_times ** power
    weights /= weights.sum()
    
    weighted_sum = (weights * log_discharges).sum()
    return pd.Series(weighted_sum, index=['weighted_average_log_discharge'])

# Calculate weighted averages for all ensemble members with a specified power for weights
def calculate_weighted_averages(df, power):
    grouped = df.groupby(['target_date', 'ensemble_member'])
    weighted_averages = grouped.apply(weighted_avg, power=power).reset_index()
    return weighted_averages

# Specify the power for weight calculation
power = -100.001  # Example: weights proportional to (lead_times)^0.1

# Calculate weighted averages
weighted_averages_all = calculate_weighted_averages(consolidated_df, power)

# Constants for USGS data
CFSToCMS_CONVERSION_FACTOR = 0.0283168466

# Define time range
start_usgs = '1979-01-01'
today = datetime.today()
end_usgs = today.strftime('%Y-%m-%d')

# Fetch daily data for the site
site_code = '11160500'
df_usgs = nwis.get_record(sites=site_code, service='dv', parameterCd='00060', statCd='00003', start=start_usgs, end=end_usgs)

# Log-transform the flow data; we add 1 to handle cases where the value is 0
df_usgs['log_discharge'] = np.log(df_usgs['00060_Mean'].astype(float) + 1)
# Keep only the relevant column
df_usgs = df_usgs[['log_discharge']]
# Reverse the log transformation to get back the discharge in cfs
df_usgs['discharge_cfs'] = np.exp(df_usgs['log_discharge']) - 1
# Convert discharge from cfs to cms
df_usgs['discharge_cms'] = df_usgs['discharge_cfs'] * CFSToCMS_CONVERSION_FACTOR
# Optionally, log-transform the discharge in cms
df_usgs['log_discharge_cms'] = np.log(df_usgs['discharge_cms'] + 1)

# Fetch metadata for the USGS site
site_info = nwis.get_record(sites=site_code, service='site')
station_name = site_info['station_nm'][0]

# Extract latitude and longitude
latitude = float(site_info['dec_lat_va'][0])
longitude = float(site_info['dec_long_va'][0])

# Combine into a single target_location tuple
target_location = (latitude, longitude)
target_lat, target_lon = target_location
print(f"The coordinates for site {site_code} are {target_location}")

# Define the consistent date range for the x-axis
start_date = weighted_averages_all['target_date'].min()
end_date = weighted_averages_all['target_date'].max()

# Create a DataFrame to store all the time series resulting from the weighting
weighted_time_series_df = weighted_averages_all.pivot(index='target_date', columns='ensemble_member', values='weighted_average_log_discharge')
weighted_time_series_df = weighted_time_series_df.reset_index()

# # # Save the DataFrame to a CSV file
# output_csv_path = '/home/jaguir26/project1_ucsc_phd/weighted_time_series.csv'
# weighted_time_series_df.to_csv(output_csv_path, index=False)

# print(f"Weighted time series saved to {output_csv_path}")

# Plotting the weighted average time series for all ensemble members along with USGS data
plt.figure(figsize=(20, 10))
ensemble_members = weighted_averages_all['ensemble_member'].unique()

for ensemble_member in ensemble_members:
    member_data = weighted_averages_all[weighted_averages_all['ensemble_member'] == ensemble_member]
    plt.plot(member_data['target_date'], member_data['weighted_average_log_discharge'], linestyle='-', alpha=0.5, label=f'Ensemble {ensemble_member}', color='orange')

# Plot USGS discharge data
plt.plot(df_usgs.index, df_usgs['log_discharge_cms'], marker='o', linestyle='--', color='forestgreen', markersize=2, alpha=1, label='USGS')

plt.xlabel('Time')
plt.ylabel('Log Discharge (cms)')
plt.title('Weighted Average Log Discharge for All Ensemble Members with USGS Data')
plt.grid(True)
plt.xlim([start_date, end_date])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


In [ ]:
import os
import pygrib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from haversine import haversine, Unit
import dataretrieval.nwis as nwis
from datetime import datetime, timedelta

# # Load the consolidated DataFrame
# consolidated_df = pd.read_csv('/home/jaguir26/projects/Project/Input/GLOFAS/consolidated_glofas_data.csv')

# Ensure 'start_forecast' column is in datetime format
consolidated_df['start_forecast'] = pd.to_datetime(consolidated_df['start_forecast'])

# Set the cutoff date
cutoff_date = '2022-12-25'

# Filter the DataFrame based on the cutoff date
consolidated_df = consolidated_df[consolidated_df['start_forecast'] <= cutoff_date]

# Calculate target dates and subtract one day
consolidated_df['target_date'] = consolidated_df['start_forecast'] + pd.to_timedelta(consolidated_df['lead_time'], unit='h') - timedelta(days=1)

# Apply log(x + 1) transformation to the discharge values
consolidated_df['log_discharge'] = np.log(consolidated_df['discharge']*2 + 1)

# Function to calculate the weighted average of the log-transformed values
def weighted_avg(group, power):
    lead_times = group['lead_time']
    log_discharges = group['log_discharge']
    
    # Ensure lead times are not zero to avoid division by zero
    if (lead_times == 0).any():
        lead_times = lead_times.replace(0, 1)
    
    weights = lead_times ** power
    weights /= weights.sum()
    
    weighted_sum = (weights * log_discharges).sum()
    return pd.Series(weighted_sum, index=['weighted_average_log_discharge'])

# Calculate weighted averages for all ensemble members with a specified power for weights
def calculate_weighted_averages(df, power):
    grouped = df.groupby(['target_date', 'ensemble_member'])
    weighted_averages = grouped.apply(weighted_avg, power=power).reset_index()
    return weighted_averages

# Specify the power for weight calculation
power = -10.001  # Example: weights proportional to (lead_times)^0.1

# Calculate weighted averages
weighted_averages_all = calculate_weighted_averages(consolidated_df, power)

# Constants for USGS data
# CFSToCMS_CONVERSION_FACTOR = 0.0283168466
CFSToCMS_CONVERSION_FACTOR = 0.0283168466

# Define time range
start_usgs = '1979-01-01'
today = datetime.today()
end_usgs = today.strftime('%Y-%m-%d')

# Fetch daily data for the site
site_code = '11160500'
df_usgs = nwis.get_record(sites=site_code, service='dv', parameterCd='00060', statCd='00003', start=start_usgs, end=end_usgs)

# Log-transform the flow data; we add 1 to handle cases where the value is 0
df_usgs['log_discharge'] = np.log(df_usgs['00060_Mean'].astype(float) + 1)
# Keep only the relevant column
df_usgs = df_usgs[['log_discharge']]
# Reverse the log transformation to get back the discharge in cfs
df_usgs['discharge_cfs'] = np.exp(df_usgs['log_discharge']) - 1
# Convert discharge from cfs to cms
df_usgs['discharge_cms'] = df_usgs['discharge_cfs'] * CFSToCMS_CONVERSION_FACTOR
# Optionally, log-transform the discharge in cms
df_usgs['log_discharge_cms'] = np.log(df_usgs['discharge_cms'] + 1)

# Fetch metadata for the USGS site
site_info = nwis.get_record(sites=site_code, service='site')
station_name = site_info['station_nm'][0]

# Extract latitude and longitude
latitude = float(site_info['dec_lat_va'][0])
longitude = float(site_info['dec_long_va'][0])

# Combine into a single target_location tuple
target_location = (latitude, longitude)
target_lat, target_lon = target_location
print(f"The coordinates for site {site_code} are {target_location}")

# Define the consistent date range for the x-axis
start_date = weighted_averages_all['target_date'].min()
end_date = weighted_averages_all['target_date'].max()

# Create a DataFrame to store all the time series resulting from the weighting
weighted_time_series_df = weighted_averages_all.pivot(index='target_date', columns='ensemble_member', values='weighted_average_log_discharge')
weighted_time_series_df = weighted_time_series_df.reset_index()

# # Save the DataFrame to a CSV file
# output_csv_path = '/home/jaguir26/project1_ucsc_phd/weighted_time_series.csv'
# weighted_time_series_df.to_csv(output_csv_path, index=False)

# print(f"Weighted time series saved to {output_csv_path}")

# Plotting the weighted average time series for all ensemble members along with USGS data
plt.figure(figsize=(20, 10))
ensemble_members = weighted_averages_all['ensemble_member'].unique()

for ensemble_member in ensemble_members:
    member_data = weighted_averages_all[weighted_averages_all['ensemble_member'] == ensemble_member]
    plt.plot(member_data['target_date'], member_data['weighted_average_log_discharge'], linestyle='-', alpha=0.5, label=f'Ensemble {ensemble_member}', color='orange')

# Plot USGS discharge data
plt.plot(df_usgs.index, df_usgs['log_discharge_cms'], marker='o', linestyle='--', color='forestgreen', markersize=2, alpha=1, label='USGS')

plt.xlabel('Time')
plt.ylabel('Log Discharge (cms)')
plt.title('Weighted Average Log Discharge for All Ensemble Members with USGS Data')
plt.grid(True)
plt.xlim([start_date, end_date])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


### Correr de nuevo el download alrededor de del verdadero target location: (37.0443931, -122.072464)
### Ajustar este notebook para crear el consolidated en este target location
### 

In [ ]:
consolidated_df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dataretrieval.nwis as nwis
from datetime import datetime, timedelta


# Load the consolidated DataFrame
consolidated_df = pd.read_csv('/home/jaguir26/projects/Project/Input/GLOFAS/consolidated_glofas_data.csv')

# Ensure 'start_forecast' column is in datetime format
consolidated_df['start_forecast'] = pd.to_datetime(consolidated_df['start_forecast'])

# Calculate target dates and subtract one day
consolidated_df['target_date'] = consolidated_df['start_forecast'] + pd.to_timedelta(consolidated_df['lead_time'], unit='h') - timedelta(days=1)

# Apply log(x + 1) transformation to the discharge values
consolidated_df['log_discharge'] = np.log(consolidated_df['discharge'] + 1)

# Define the total number of ensemble members
total_ensembles = consolidated_df['ensemble_member'].nunique()

# Function to calculate the custom weighted average of the log-transformed values
def custom_weighted_avg(group, a_j):
    lead_times = group['lead_time']
    log_discharges = group['log_discharge']
    
    # Ensure lead times are not zero to avoid division by zero
    if (lead_times == 0).any():
        lead_times = lead_times.replace(0, 1)
    
    weights = 1 / (lead_times ** a_j)
    weights /= weights.sum()
    
    weighted_sum = (weights * log_discharges).sum()
    return pd.Series(weighted_sum, index=['weighted_average_log_discharge'])

# Calculate weighted averages for all ensemble members with custom weights
def calculate_custom_weighted_averages(df, total_ensembles):
    results = []
    
    for ensemble_member in range(0, total_ensembles):
        # a_j = 0.5 + (ensemble_member / (total_ensembles + 40))
        a_j = 0.5
        member_df = df[df['ensemble_member'] == ensemble_member]
        grouped = member_df.groupby('target_date', as_index=False)
        weighted_averages = grouped.apply(custom_weighted_avg, a_j=a_j).reset_index(drop=True)
        weighted_averages['ensemble_member'] = ensemble_member
        results.append(weighted_averages)
    
    return pd.concat(results, ignore_index=True)

# Calculate custom weighted averages
weighted_averages_all = calculate_custom_weighted_averages(consolidated_df, total_ensembles)

# Constants for USGS data
CFSToCMS_CONVERSION_FACTOR = 0.0283168466

# Define time range
start_usgs = '1979-01-01'
today = datetime.today()
end_usgs = today.strftime('%Y-%m-%d')

# Fetch daily data for the site
site_code = '11160500'
df_usgs = nwis.get_record(sites=site_code, service='dv', parameterCd='00060', statCd='00003', start=start_usgs, end=end_usgs)

# Log-transform the flow data; we add 1 to handle cases where the value is 0
df_usgs['log_discharge'] = np.log(df_usgs['00060_Mean'].astype(float) + 1)
# Keep only the relevant column
df_usgs = df_usgs[['log_discharge']]
# Reverse the log transformation to get back the discharge in cfs
df_usgs['discharge_cfs'] = np.exp(df_usgs['log_discharge']) - 1
# Convert discharge from cfs to cms
df_usgs['discharge_cms'] = df_usgs['discharge_cfs'] * CFSToCMS_CONVERSION_FACTOR
# Optionally, log-transform the discharge in cms
df_usgs['log_discharge_cms'] = np.log(df_usgs['discharge_cms'] + 1)

# Fetch metadata for the USGS site
site_info = nwis.get_record(sites=site_code, service='site')
station_name = site_info['station_nm'][0]

# Extract latitude and longitude
latitude = float(site_info['dec_lat_va'][0])
longitude = float(site_info['dec_long_va'][0])

# Combine into a single target_location tuple
target_location = (latitude, longitude)
target_lat, target_lon = target_location
print(f"The coordinates for site {site_code} are {target_location}")

# Define the consistent date range for the x-axis
start_date = weighted_averages_all['target_date'].min()
end_date = weighted_averages_all['target_date'].max()

# Create a DataFrame to store all the time series resulting from the custom weighting
weighted_time_series_df = weighted_averages_all.pivot(index='target_date', columns='ensemble_member', values='weighted_average_log_discharge')
weighted_time_series_df = weighted_time_series_df.reset_index()

# Save the DataFrame to a CSV file
output_csv_path = '/home/jaguir26/project1_ucsc_phd/weighted_time_series_custom.csv'
weighted_time_series_df.to_csv(output_csv_path, index=False)

print(f"Weighted time series saved to {output_csv_path}")


In [ ]:
weighted_time_series_df[weighted_time_series_df['target_date']=='2022-12-26']

In [ ]:

# Plotting the weighted average time series for all ensemble members along with USGS data
plt.figure(figsize=(20, 10))
ensemble_members = weighted_averages_all['ensemble_member'].unique()

for ensemble_member in ensemble_members:
    member_data = weighted_averages_all[weighted_averages_all['ensemble_member'] == ensemble_member]
    plt.plot(member_data['target_date'], member_data['weighted_average_log_discharge'], linestyle='-', alpha=0.5, label=f'Ensemble {ensemble_member}')

# Plot USGS discharge data
plt.plot(df_usgs.index, df_usgs['log_discharge_cms'], marker='o', linestyle='--', color='forestgreen', markersize=2, alpha=1, label='USGS')

plt.xlabel('Time')
plt.ylabel('Log Discharge (cms)')
plt.title('Custom Weighted Average Log Discharge for All Ensemble Members with USGS Data')
plt.grid(True)
plt.xlim([start_date, end_date])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dataretrieval.nwis as nwis
from datetime import datetime, timedelta

# Load the consolidated DataFrame
consolidated_df = pd.read_csv('/home/jaguir26/projects/Project/Input/GLOFAS/consolidated_glofas_data.csv')

# Ensure 'start_forecast' column is in datetime format
consolidated_df['start_forecast'] = pd.to_datetime(consolidated_df['start_forecast'])

# Set the cutoff date
cutoff_date = '2022-12-25'

# Filter the DataFrame based on the cutoff date
consolidated_df = consolidated_df[consolidated_df['start_forecast'] <= cutoff_date]

# Calculate target dates and subtract one day
consolidated_df['target_date'] = consolidated_df['start_forecast'] + pd.to_timedelta(consolidated_df['lead_time'], unit='h') - timedelta(days=1)

# Apply log(x + 1) transformation to the discharge values
consolidated_df['log_discharge'] = np.log(consolidated_df['discharge'] + 1)

# Define the total number of ensemble members
total_ensembles = consolidated_df['ensemble_member'].nunique()

# Function to calculate the custom weighted average of the log-transformed values
def custom_weighted_avg(group, a_j):
    lead_times = group['lead_time']
    log_discharges = group['log_discharge']
    
    # Ensure lead times are not zero to avoid division by zero
    if (lead_times == 0).any():
        lead_times = lead_times.replace(0, 1)
    
    weights = 1 / (lead_times ** a_j)
    weights /= weights.sum()
    
    weighted_sum = (weights * log_discharges).sum()
    return pd.Series(weighted_sum, index=['weighted_average_log_discharge'])

# Calculate weighted averages for all ensemble members with custom weights
def calculate_custom_weighted_averages(df, total_ensembles):
    results = []
    
    for ensemble_member in range(0, total_ensembles):
        # a_j = 0.5 + (ensemble_member / (total_ensembles + 40))
        a_j = 3.01
        member_df = df[df['ensemble_member'] == ensemble_member]
        grouped = member_df.groupby('target_date', as_index=False)
        weighted_averages = grouped.apply(custom_weighted_avg, a_j=a_j).reset_index(drop=True)
        weighted_averages['ensemble_member'] = ensemble_member
        results.append(weighted_averages)
    
    return pd.concat(results, ignore_index=True)

# Calculate custom weighted averages
weighted_averages_all = calculate_custom_weighted_averages(consolidated_df, total_ensembles)

# Constants for USGS data
CFSToCMS_CONVERSION_FACTOR = 0.0283168466

# Define time range
start_usgs = '1979-01-01'
today = datetime.today()
end_usgs = today.strftime('%Y-%m-%d')

# Fetch daily data for the site
site_code = '11160500'
df_usgs = nwis.get_record(sites=site_code, service='dv', parameterCd='00060', statCd='00003', start=start_usgs, end=end_usgs)

# Log-transform the flow data; we add 1 to handle cases where the value is 0
df_usgs['log_discharge'] = np.log(df_usgs['00060_Mean'].astype(float) + 1)
# Keep only the relevant column
df_usgs = df_usgs[['log_discharge']]
# Reverse the log transformation to get back the discharge in cfs
df_usgs['discharge_cfs'] = np.exp(df_usgs['log_discharge']) - 1
# Convert discharge from cfs to cms
df_usgs['discharge_cms'] = df_usgs['discharge_cfs'] * CFSToCMS_CONVERSION_FACTOR
# Optionally, log-transform the discharge in cms
df_usgs['log_discharge_cms'] = np.log(df_usgs['discharge_cms'] + 1)

# Fetch metadata for the USGS site
site_info = nwis.get_record(sites=site_code, service='site')
station_name = site_info['station_nm'][0]

# Extract latitude and longitude
latitude = float(site_info['dec_lat_va'][0])
longitude = float(site_info['dec_long_va'][0])

# Combine into a single target_location tuple
target_location = (latitude, longitude)
target_lat, target_lon = target_location
print(f"The coordinates for site {site_code} are {target_location}")

# Define the consistent date range for the x-axis
start_date = weighted_averages_all['target_date'].min()
end_date = weighted_averages_all['target_date'].max()

# Create a DataFrame to store all the time series resulting from the custom weighting
weighted_time_series_df = weighted_averages_all.pivot(index='target_date', columns='ensemble_member', values='weighted_average_log_discharge')
weighted_time_series_df = weighted_time_series_df.reset_index()

# Save the DataFrame to a CSV file
output_csv_path = '/home/jaguir26/project1_ucsc_phd/weighted_time_series_custom.csv'
weighted_time_series_df.to_csv(output_csv_path, index=False)

print(f"Weighted time series saved to {output_csv_path}")

# Plotting the weighted average time series for all ensemble members along with USGS data
plt.figure(figsize=(20, 10))
ensemble_members = weighted_averages_all['ensemble_member'].unique()

for ensemble_member in ensemble_members:
    member_data = weighted_averages_all[weighted_averages_all['ensemble_member'] == ensemble_member]
    plt.plot(member_data['target_date'], member_data['weighted_average_log_discharge'], linestyle='-', alpha=0.5, label=f'Ensemble {ensemble_member}')

# Plot USGS discharge data
plt.plot(df_usgs.index, df_usgs['log_discharge_cms'], marker='o', linestyle='--', color='forestgreen', markersize=2, alpha=1, label='USGS')

plt.xlabel('Time')
plt.ylabel('Log Discharge (cms)')
plt.title('Custom Weighted Average Log Discharge for All Ensemble Members with USGS Data')
plt.grid(True)
plt.xlim([start_date, end_date])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


In [ ]:
import os
import cdsapi
from datetime import datetime, timedelta
import numpy as np
from haversine import haversine, Unit
import pygrib
import pandas as pd

os.chdir('/home/jaguir26/projects/')

def create_directory_NAME(base_path, folder_name):
    directory_path = os.path.join(base_path, folder_name)
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
    return directory_path

base_path = os.getcwd()
directory_path_project = create_directory_NAME(base_path, 'Project')

base_path = directory_path_project
directory_path_input = create_directory_NAME(base_path, 'Input')
directory_path_output = create_directory_NAME(base_path, 'Output')

base_path = directory_path_input
directory_path_retro = create_directory_NAME(base_path, 'Retrospective_Analysis')
directory_path_retro_glofas = create_directory_NAME(base_path, 'GLOFAS')

# Define the base directory for GloFAS medium-range forecasts
base_directory_glofas_frsc_med = create_directory_NAME(directory_path_retro_glofas, 'Medium_Range')

# Function to convert date format
def convert_date_format(date_str):
    return date_str.replace("-", "")

def choose_hydrological_model(start_forecast):
    lisflood_start_date = datetime.strptime('2021-05-26', '%Y-%m-%d')
    forecast_date = datetime.strptime(start_forecast, '%Y-%m-%d')
    if forecast_date < lisflood_start_date:
        return 'htessel_lisflood'
    else:
        return 'lisflood'

def convert_longitude_to_neg_180_180(lon):
    if lon > 180:
        return lon - 360
    return lon

def convert_longitude_to_0_360(lon):
    if lon < 0:
        return lon + 360
    return lon

def retrieve_glofas_data_medium(
    start_forecast,
    target_location,
    buffer=2,
    system_version='operational',
    product_type=['control_forecast'],
    variable='river_discharge_in_the_last_24_hours',
    leadtime_hour=[str(i) for i in range(24, 48, 24)],
    base_directory_glofas_frsc_med=base_directory_glofas_frsc_med
):
    if datetime.strptime(start_forecast, '%Y-%m-%d') < datetime(2019, 11, 11):
        print("No data available before Nov 11, 2019 for medium-range forecasts.")
        return None
    
    hydrological_model = choose_hydrological_model(start_forecast)
    dt = datetime.strptime(start_forecast, '%Y-%m-%d')
    year, month, day = dt.year, dt.month, dt.day
    latitude, longitude = target_location
    area = [latitude + buffer, longitude - buffer, latitude - buffer, longitude + buffer]
    short_product_type = '_'.join(''.join(word[0] for word in item.split('_')) for item in product_type)
    short_variable = ''.join(word[0] for word in variable.split('_'))
    file_name_elements = [
        system_version[:3], 
        hydrological_model[:3], 
        short_product_type,
        short_variable,
        str(year), str(month).zfill(2), str(day).zfill(2),
        f"lt_{leadtime_hour[0]}_to_{leadtime_hour[-1]}", 
        f"area_{round(area[0], 2)}_{round(area[1], 2)}_{round(area[2], 2)}_{round(area[3], 2)}"
    ]
    file_name = '_'.join(file_name_elements) + '.grib'
    
    # Create a directory for this specific start forecast date
    date_directory = create_directory_NAME(base_directory_glofas_frsc_med, start_forecast)
    output_file = os.path.join(date_directory, file_name)
    
    if os.path.exists(output_file):
        print(f"File already exists: {output_file}")
        return output_file
    
    c = cdsapi.Client()
    retrieval_params = {
        'system_version': system_version,
        'hydrological_model': hydrological_model,
        'product_type': product_type,
        'variable': variable,
        'year': str(year),
        'month': str(month).zfill(2),
        'day': str(day).zfill(2),
        'leadtime_hour': leadtime_hour,
        'format': 'grib',
        'area': area
    }
    c.retrieve('cems-glofas-forecast', retrieval_params, output_file)
    print(f"Retrieval completed. Output file saved to: {output_file}")
    return output_file

def process_grib_file(file_path):
    if file_path is None or not os.path.exists(file_path):
        print(f"File does not exist or path is None: {file_path}")
        return None, None, None

    grbs = pygrib.open(file_path)
    unique_lead_times = set()
    unique_ensemble_members = set()

    for grb in grbs:
        unique_lead_times.add(grb['forecastTime'])
        unique_ensemble_members.add(grb['perturbationNumber'])

    grbs.seek(0)
    return grbs, unique_lead_times, unique_ensemble_members

def find_all_coordinates(grbs):
    if grbs is None:
        return None

    all_coordinates = []

    for grb in grbs:
        if grb['perturbationNumber'] == 0:  # Considering control forecast only
            latitudes, longitudes = grb.latlons()
            for i in range(len(latitudes)):
                for j in range(len(longitudes[0])):
                    coord = (latitudes[i][j], convert_longitude_to_neg_180_180(longitudes[i][j]))
                    all_coordinates.append((coord, i, j))
            break
    
    return all_coordinates


# Loop over the date range and verify coordinates before downloading data
date_range = pd.date_range(start='2023-01-06', end='2023-01-06', freq='D')
# target_location = (37.0443931, -122.072464)  # Example coordinates, replace with actual coordinates
target_location = (34.05, -118.25)
buffer = 1  

for single_date in date_range:
    start_forecast = single_date.strftime('%Y-%m-%d')
    grib_file_path = retrieve_glofas_data_medium(start_forecast=start_forecast, target_location=target_location, buffer=buffer)
    grbs, unique_lead_times, unique_ensemble_members = process_grib_file(grib_file_path)
    
    all_coordinates = find_all_coordinates(grbs)
    if all_coordinates:
        print(f"Date: {start_forecast}")
        print("All Coordinates:")
        for coord, i, j in all_coordinates:
            print(f"Coord: {coord}, Index: ({i},{j})")
    else:
        print(f"No coordinates found within the buffer for date: {start_forecast}")


In [ ]:
import os
import cdsapi
from datetime import datetime, timedelta
import numpy as np
from haversine import haversine, Unit
import pygrib
import pandas as pd

def find_all_coordinates_and_log_values(grbs):
    if grbs is None:
        return None, None

    all_coordinates = []
    log_streamflow_values = []

    for grb in grbs:
        if grb['perturbationNumber'] == 0:  # Considering control forecast only
            latitudes, longitudes = grb.latlons()
            for i in range(len(latitudes)):
                for j in range(len(longitudes[0])):
                    coord = (latitudes[i][j], convert_longitude_to_neg_180_180(longitudes[i][j]))
                    value = grb.values[i, j]
                    if value > 0:  # Take the log only if the value is positive
                        log_value = np.log(value+1)
                    else:
                        log_value = np.nan  # Handle non-positive values
                    all_coordinates.append((coord, i, j))
                    log_streamflow_values.append(log_value)
            break
    
    return all_coordinates, log_streamflow_values


In [ ]:
import os
import cdsapi
from datetime import datetime, timedelta
import numpy as np
from haversine import haversine, Unit
import pygrib
import pandas as pd

# Loop over the date range and plot the grid for the log streamflow values
date_range = pd.date_range(start='2023-01-05', end='2023-01-05', freq='D')
# target_location = (37.0443931, -122.072464)  # Example coordinates, replace with actual coordinates
target_location = (34.05, -118.25)
buffer = 1  # Adjust the buffer as needed

for single_date in date_range:
    start_forecast = single_date.strftime('%Y-%m-%d')
    grib_file_path = retrieve_glofas_data_medium(start_forecast=start_forecast, target_location=target_location, buffer=buffer)
    grbs, unique_lead_times, unique_ensemble_members = process_grib_file(grib_file_path)
    
    all_coordinates, log_streamflow_values = find_all_coordinates_and_log_values(grbs)
    
    if all_coordinates:
        print(f"Date: {start_forecast}")
        print("Plotting log of streamflow values on a grid...")

        # Extract latitude and longitude indices for the grid
        lat_indices = sorted(set(coord[1] for coord in all_coordinates))
        lon_indices = sorted(set(coord[2] for coord in all_coordinates))

        # Create a 2D array to store log streamflow values
        log_streamflow_grid = np.full((len(lat_indices), len(lon_indices)), np.nan)

        for (coord, i, j), log_value in zip(all_coordinates, log_streamflow_values):
            lat_idx = lat_indices.index(i)
            lon_idx = lon_indices.index(j)
            log_streamflow_grid[lat_idx, lon_idx] = log_value

        # Plot the grid
        plt.figure(figsize=(10, 8))
        plt.imshow(log_streamflow_grid, origin='upper', cmap='viridis', extent=[lon_indices[0], lon_indices[-1], lat_indices[0], lat_indices[-1]])
        plt.colorbar(label='Log(Streamflow) (units)')
        plt.title(f'Log of Streamflow Values on {start_forecast}')
        plt.xlabel('Longitude Index')
        plt.ylabel('Latitude Index')
        plt.show()
    else:
        print(f"No coordinates found within the buffer for date: {start_forecast}")


In [ ]:
import os
import cdsapi
from datetime import datetime, timedelta
import numpy as np
from haversine import haversine, Unit
import pygrib
import pandas as pd

# Loop over the date range and plot the grid for the log streamflow values
date_range = pd.date_range(start='2023-01-01', end='2023-01-01', freq='D')
# target_location = (37.0443931, -122.072464)  # Example coordinates, replace with actual coordinates
target_location = (34.05, -118.25)
buffer = 1  # Adjust the buffer as needed

for single_date in date_range:
    start_forecast = single_date.strftime('%Y-%m-%d')
    grib_file_path = retrieve_glofas_data_medium(start_forecast=start_forecast, target_location=target_location, buffer=buffer)
    grbs, unique_lead_times, unique_ensemble_members = process_grib_file(grib_file_path)
    
    all_coordinates, log_streamflow_values = find_all_coordinates_and_log_values(grbs)
    
    if all_coordinates:
        print(f"Date: {start_forecast}")
        print("Plotting log of streamflow values on a grid...")

        # Extract latitude and longitude indices for the grid
        lat_indices = sorted(set(coord[1] for coord in all_coordinates))
        lon_indices = sorted(set(coord[2] for coord in all_coordinates))

        # Create a 2D array to store log streamflow values
        log_streamflow_grid = np.full((len(lat_indices), len(lon_indices)), np.nan)

        # Variables to track the maximum value and its location
        max_log_value = -np.inf
        max_log_coord = None
        max_log_index = (None, None)

        for (coord, i, j), log_value in zip(all_coordinates, log_streamflow_values):
            lat_idx = lat_indices.index(i)
            lon_idx = lon_indices.index(j)
            log_streamflow_grid[lat_idx, lon_idx] = log_value
            
            # Update the maximum value and its location if the current value is greater
            if log_value > max_log_value:
                max_log_value = log_value
                max_log_coord = coord
                max_log_index = (lat_idx, lon_idx)

        # Plot the grid
        plt.figure(figsize=(10, 8))
        plt.imshow(log_streamflow_grid, origin='upper', cmap='viridis', extent=[lon_indices[0], lon_indices[-1], lat_indices[0], lat_indices[-1]])
        plt.colorbar(label='Log(Streamflow) (units)')
        plt.title(f'Log of Streamflow Values on {start_forecast}')
        plt.xlabel('Longitude Index')
        plt.ylabel('Latitude Index')
        plt.scatter([max_log_index[1]], [max_log_index[0]], color='red', marker='x', s=100, label='Max Value')
        plt.legend()
        plt.show()

        # Print the maximum log value and its location
        print(f"Maximum log streamflow value: {max_log_value}")
        print(f"Location of maximum value: {max_log_coord}")
    else:
        print(f"No coordinates found within the buffer for date: {start_forecast}")
